In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
%sql
create or replace temporary view orders_bronze_changes
AS 
select * from table_changes('retaildb.orders_bronze', 1) where order_id > 0
AND customer_id > 0 and order_status IN ("CLOSED", "PENDING_PAYMENT");

In [0]:
%sql
merge into retaildb.orders_silver tgt
using orders_bronze_changes src on tgt.order_id = src.order_id
when matched
then
update set tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
WHEN not matched
then
insert(order_id, order_date, customer_id, order_status, createdon, modifiedon) values(order_id, order_date, customer_id, order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

In [0]:
%sql
insert overwrite table retaildb.orders_gold
select customer_id, order_status, order_year, count(order_id) as num_orders
from retaildb.orders_silver group by 1, 2, 3;
-- let's run this